# 📊 Financial Analysis Platform
## Macro & Micro Analysis — Investment Decision Support
> **Tech-heavy portfolio** (semis + AI + clean energy) · 3-year history · 3 optimization strategies

## 0. Setup & Dependencias

In [ ]:
# Install all required packages
!pip install -q yfinance prophet statsmodels xgboost scikit-learn PyPortfolioOpt
!pip install -q pandas numpy scipy matplotlib seaborn plotly
print('✅ All packages installed')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import json

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Try plotly for interactive charts
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY = True
except ImportError:
    PLOTLY = False

print(f'NumPy {np.__version__} | Pandas {pd.__version__} | Plotly: {PLOTLY}')

## 1. Data Acquisition

In [ ]:
import yfinance as yf

TICKERS = ['NVDA', 'MSFT', 'GOOGL', 'SOXX', 'SMH', 'TAN', 'NLR', 'URNM']
BENCHMARKS = ['SPY', 'QQQ']
ALL_TICKERS = TICKERS + BENCHMARKS

END   = datetime.today().strftime('%Y-%m-%d')
START = (datetime.today() - relativedelta(years=3)).strftime('%Y-%m-%d')

print(f'Downloading {len(ALL_TICKERS)} tickers from {START} to {END}...')
raw = yf.download(ALL_TICKERS, start=START, end=END, auto_adjust=True, progress=False)
prices = raw['Close'].dropna()
returns = prices[TICKERS].pct_change().dropna()

print(f'✅ Downloaded: {prices.shape[0]} trading days × {prices.shape[1]} assets')
prices.tail(3)

## 2. Exploratory Data Analysis

In [ ]:
# Annualized return and volatility
ann_ret = returns.mean() * 252
ann_vol = returns.std() * np.sqrt(252)
sharpe  = (ann_ret - 0.0425) / ann_vol

stats = pd.DataFrame({'Ann. Return': ann_ret, 'Ann. Vol': ann_vol,
                      'Sharpe': sharpe}).round(4)
print('=== Individual Asset Statistics ===')
print(stats.to_string())

# Correlation matrix
corr = returns.corr()
print('\n=== Correlation Matrix ===')
print(corr.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Normalized prices
ax = axes[0, 0]
(prices[TICKERS] / prices[TICKERS].iloc[0] * 100).plot(ax=ax)
ax.set_title('Normalized Price Performance (Base 100)')
ax.set_ylabel('Index Value')
ax.legend(fontsize=8)

# Return distribution
ax = axes[0, 1]
returns[TICKERS].plot(kind='box', ax=ax, rot=45)
ax.set_title('Daily Return Distribution')
ax.set_ylabel('Daily Return')

# Correlation heatmap
ax = axes[1, 0]
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=ax, annot_kws={'size': 8})
ax.set_title('Asset Correlation Matrix')

# Risk-Return scatter
ax = axes[1, 1]
ax.scatter(ann_vol * 100, ann_ret * 100, s=100, zorder=5)
for ticker in TICKERS:
    ax.annotate(ticker, (ann_vol[ticker] * 100, ann_ret[ticker] * 100),
               xytext=(5, 5), textcoords='offset points', fontsize=9)
ax.set_xlabel('Annual Volatility (%)')
ax.set_ylabel('Annual Return (%)')
ax.set_title('Risk-Return Space')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved')

## 3. Portfolio Optimization

In [ ]:
from pypfopt import EfficientFrontier, objective_functions, risk_models, expected_returns

mu  = expected_returns.mean_historical_return(prices[TICKERS])
cov = risk_models.sample_cov(prices[TICKERS])
RF  = 0.0425

portfolios = {}

# P1: Minimum Volatility
ef = EfficientFrontier(mu, cov, weight_bounds=(0, 0.40))
ef.add_objective(objective_functions.L2_reg, gamma=0.1)
ef.min_volatility()
w1 = ef.clean_weights()
p1 = ef.portfolio_performance(risk_free_rate=RF)
portfolios['Min Volatilidad'] = {'weights': dict(w1), 'perf': p1}

# P2: Max Sharpe (Volatilidad Media)
ef = EfficientFrontier(mu, cov, weight_bounds=(0, 0.40))
ef.add_objective(objective_functions.L2_reg, gamma=0.1)
ef.max_sharpe(risk_free_rate=RF)
w2 = ef.clean_weights()
p2 = ef.portfolio_performance(risk_free_rate=RF)
portfolios['Volatilidad Media'] = {'weights': dict(w2), 'perf': p2}

# P3: Maximum Risk (low risk-aversion utility)
ef = EfficientFrontier(mu, cov, weight_bounds=(0, 0.60))
ef.max_quadratic_utility(risk_aversion=0.05)
w3 = ef.clean_weights()
p3 = ef.portfolio_performance(risk_free_rate=RF)
portfolios['Máximo Riesgo'] = {'weights': dict(w3), 'perf': p3}

print('=== Portfolio Optimization Results ===')
for name, data in portfolios.items():
    ret, vol, sharpe = data['perf']
    print(f'\n{name}:')
    print(f'  Return: {ret:.2%} | Vol: {vol:.2%} | Sharpe: {sharpe:.3f}')
    w_nonzero = {k: v for k, v in data['weights'].items() if v > 0.001}
    print(f'  Weights: {w_nonzero}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = plt.cm.Set3(np.linspace(0, 1, len(TICKERS)))

for ax, (name, data) in zip(axes, portfolios.items()):
    w = {k: v for k, v in data['weights'].items() if v > 0.001}
    ax.pie(w.values(), labels=w.keys(), autopct='%1.1f%%',
           colors=colors[:len(w)], startangle=90)
    ret, vol, sharpe = data['perf']
    ax.set_title(f'{name}\nReturn: {ret:.1%} | Vol: {vol:.1%} | Sharpe: {sharpe:.2f}',
                 fontsize=10)

plt.suptitle('Portfolio Compositions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('portfolio_compositions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Macro Analysis

In [ ]:
# Download macro indicators
macro_tickers = {
    'TLT': '20Y Treasury ETF',
    'GLD': 'Gold ETF',
    'DXY=X': 'USD Index',
    'VIX': 'Volatility Index',
}

try:
    macro_raw = yf.download(list(macro_tickers.keys()), start=START, end=END,
                            auto_adjust=True, progress=False)
    macro = macro_raw['Close'].dropna(how='all')
    macro.columns = [macro_tickers.get(c, c) for c in macro.columns]
    
    print('=== Macro Indicators Summary ===')
    print(macro.describe().round(2).to_string())
    
    # Correlation with portfolio
    spy_ret = prices['SPY'].pct_change().dropna()
    for col in macro.columns:
        if col in macro:
            macro_ret = macro[col].pct_change().dropna()
            common = spy_ret.index.intersection(macro_ret.index)
            if len(common) > 50:
                corr_val = spy_ret[common].corr(macro_ret[common])
                print(f'SPY vs {col}: {corr_val:.3f}')
except Exception as e:
    print(f'Macro data unavailable: {e}')

## 5. Ensemble Forecast + Monte Carlo

In [ ]:
import sys
sys.path.insert(0, '.')  # add current dir to path

# If running standalone (not from cloned repo), define inline
try:
    from ensemble_forecast import run_full_pipeline, portfolio_nav, build_three_portfolios
    print('✅ Loaded ensemble_forecast module')
except ImportError:
    print('⚠️  ensemble_forecast.py not found — run the colab_generate_forecasts notebook instead')
    raise

In [ ]:
# Run the full pipeline (downloads data, optimizes, forecasts, Monte Carlo)
results = run_full_pipeline(
    H=21,          # 21-day forecast horizon
    n_sims=10_000, # Monte Carlo simulations
    years=3,       # years of historical data
    out_json='forecast_results.json'  # output file
)
print('\n✅ Pipeline complete!')

## 6. Results & Visualization

In [ ]:
# Load and display results
with open('forecast_results.json') as f:
    res = json.load(f)

print(f"Data mode: {res['meta']['data_mode']}  |  Generated: {res['meta']['generated']}")
print(f"Rows: {res['meta']['rows']}  |  MC Sims: {res['meta']['n_sims']}  |  Horizon: {res['meta']['horizon_days']}d")
print()

# Summary table
rows = []
for name, p in res['portfolios'].items():
    mc = p['monte_carlo']
    pr = mc['probabilities']
    comp = p['composition']
    rows.append({
        'Portfolio': name,
        'Sharpe': comp['sharpe'],
        'Ann.Vol': f"{comp['exp_vol']:.1%}",
        'Ens.Ret%': f"{mc['ensemble_return_pct']:+.2f}%",
        'P(+)': f"{pr['P_positivo']:.1%}",
        'P(>5%)': f"{pr['P_ret_mayor_5pct']:.1%}",
        'VaR95': f"{mc['VaR_95_pct']:.2f}%",
        'CVaR95': f"{mc['CVaR_95_pct']:.2f}%",
    })

df_summary = pd.DataFrame(rows).set_index('Portfolio')
print('=== Portfolio Results Summary ===')
print(df_summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (name, p) in zip(axes, res['portfolios'].items()):
    mc = p['monte_carlo']
    bands = mc['bands']
    fc = p['ensemble_forecast']
    H = len(fc)
    days = list(range(H))
    
    # Monte Carlo bands
    ax.fill_between(days, [bands['P5']] * H, [bands['P95']] * H,
                    alpha=0.15, color='blue', label='P5-P95')
    ax.fill_between(days, [bands['P25']] * H, [bands['P75']] * H,
                    alpha=0.3, color='blue', label='P25-P75')
    
    # Ensemble forecast
    ax.plot(days, fc, 'b-', linewidth=2, label='Ensemble')
    ax.axhline(mc['P0'], color='gray', linestyle='--', label=f"P0={mc['P0']:.1f}")
    ax.axhline(mc['mc_median_terminal'], color='orange', linestyle=':',
               label=f"MC Median")
    
    pr = mc['probabilities']
    ax.set_title(f"{name}\nP(+)={pr['P_positivo']:.1%}  VaR95={mc['VaR_95_pct']:.1f}%",
                fontsize=10)
    ax.set_xlabel('Trading Days')
    ax.set_ylabel('NAV')
    ax.legend(fontsize=8)

plt.suptitle('21-Day Ensemble Forecast + Monte Carlo Bands', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('forecast_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Terminal distribution histograms
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (name, p) in zip(axes, res['portfolios'].items()):
    mc = p['monte_carlo']
    sample = p.get('mc_sample', [])
    if sample:
        ax.hist(sample, bins=80, color='steelblue', alpha=0.7, edgecolor='none')
        ax.axvline(mc['P0'], color='black', linewidth=2, label=f"P0={mc['P0']:.1f}")
        ax.axvline(mc['ensemble_terminal'], color='red', linewidth=2,
                   label=f"Ensemble={mc['ensemble_terminal']:.1f}")
        ax.axvline(mc['mc_median_terminal'], color='orange', linewidth=2,
                   linestyle='--', label=f"MC Med={mc['mc_median_terminal']:.1f}")
        bands = mc['bands']
        ax.axvline(bands['P5'], color='red', linewidth=1, linestyle=':',
                   label=f"P5={bands['P5']:.1f}")
        ax.axvline(bands['P95'], color='green', linewidth=1, linestyle=':',
                   label=f"P95={bands['P95']:.1f}")
    
    pr = mc['probabilities']
    ax.set_title(f"{name}\nP(+)={pr['P_positivo']:.1%}  CVaR95={mc['CVaR_95_pct']:.1f}%",
                fontsize=10)
    ax.set_xlabel('Terminal NAV')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)

plt.suptitle('Terminal NAV Distribution (Monte Carlo 10,000 sims)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('terminal_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Performance Analysis

In [ ]:
print('=== Backtest Error by Model and Portfolio ===\n')

model_names = ['Prophet', 'ARIMAX', 'XGBoost', 'Holt-Winters']

for name, p in res['portfolios'].items():
    ens = p['ensemble']
    bt = ens.get('backtest_error', {})
    weights = ens.get('weights', {})
    print(f'{name}:')
    for model in model_names:
        err = bt.get(model, {})
        w = weights.get(model, 0)
        if 'rmse' in err:
            print(f'  {model:<15} RMSE={err["rmse"]:.2f}  MAPE={err["mape"]:.2f}%  weight={w:.4f}')
        elif 'error' in err:
            print(f'  {model:<15} ERROR: {err["error"]}')
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (name, p) in zip(axes, res['portfolios'].items()):
    fc_dict = p.get('model_forecasts', {})
    ens_fc  = p['ensemble_forecast']
    H = len(ens_fc)
    
    colors_map = {'Prophet': 'blue', 'ARIMAX': 'green',
                  'XGBoost': 'orange', 'Holt-Winters': 'purple'}
    for model, fc in fc_dict.items():
        ax.plot(fc, linestyle='--', alpha=0.7,
                color=colors_map.get(model, 'gray'), label=model)
    ax.plot(ens_fc, 'k-', linewidth=2.5, label='Ensemble')
    mc = p['monte_carlo']
    ax.axhline(mc['P0'], color='red', linestyle=':', alpha=0.5, label=f"P0")
    ax.set_title(f'{name}\nModel Comparison', fontsize=10)
    ax.set_xlabel('Days')
    ax.set_ylabel('NAV')
    ax.legend(fontsize=8)

plt.suptitle('Individual Model Forecasts vs Ensemble', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Risk Analysis

In [ ]:
print('=== Risk Metrics Summary ===\n')
print(f'{"Portfolio":<20} {"Sharpe":>8} {"Ann.Vol":>10} {"VaR95":>8} {"CVaR95":>8} {"P(+)":>8} {"P(>5%)":>8}')
print('-' * 75)

for name, p in res['portfolios'].items():
    mc   = p['monte_carlo']
    pr   = mc['probabilities']
    comp = p['composition']
    print(f"{name:<20} {comp['sharpe']:>8.3f} {comp['exp_vol']:>10.1%} "
          f"{mc['VaR_95_pct']:>7.2f}% {mc['CVaR_95_pct']:>7.2f}% "
          f"{pr['P_positivo']:>7.1%} {pr['P_ret_mayor_5pct']:>8.1%}")

In [ ]:
# Rolling drawdown analysis
def max_drawdown(series):
    roll_max = series.cummax()
    drawdown = (series - roll_max) / roll_max
    return drawdown

# Build NAV series for each portfolio
from ensemble_forecast import portfolio_nav

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

nav_series = {}
for name, p in res['portfolios'].items():
    comp = p['composition']
    nav = portfolio_nav(comp['weights'], prices)
    nav_series[name] = nav

# NAV plot
ax = axes[0]
for name, nav in nav_series.items():
    nav.plot(ax=ax, label=name)
ax.set_title('Portfolio NAV (Historical)')
ax.set_ylabel('NAV (Base 100)')
ax.legend()

# Drawdown plot
ax = axes[1]
for name, nav in nav_series.items():
    dd = max_drawdown(nav)
    (dd * 100).plot(ax=ax, label=name)
ax.set_title('Portfolio Drawdown (%)')
ax.set_ylabel('Drawdown (%)')
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('drawdown_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Multi-Agent Interpretation

In [ ]:
# Load and display the multi-agent interpretations
try:
    with open('AGENT_INTERPRETATIONS.md', 'r') as f:
        content = f.read()
    # Display in notebook
    from IPython.display import Markdown, display
    display(Markdown(content))
except FileNotFoundError:
    print('AGENT_INTERPRETATIONS.md not found')
    print('Run the multi-agent analysis pipeline to generate it.')

## 10. Export Results

In [ ]:
import os

# List generated files
artifacts = [
    'forecast_results.json',
    'eda_analysis.png',
    'portfolio_compositions.png',
    'forecast_visualization.png',
    'terminal_distributions.png',
    'model_comparison.png',
    'drawdown_analysis.png',
]

print('=== Generated Artifacts ===')
for f in artifacts:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    print(f"  {'✅' if exists else '❌'} {f} ({size:,} bytes)")

# Download all from Colab
try:
    from google.colab import files
    for f in artifacts:
        if os.path.exists(f):
            files.download(f)
    print('\n✅ Downloads initiated')
except ImportError:
    print('\n(Not in Colab — files are in the current directory)')

---
*Este análisis es exclusivamente educativo y no constituye asesoramiento financiero.*

In [ ]:
print('=' * 80)
print('  FINANCIAL ANALYSIS PLATFORM — Complete')
print('  Portfolios: Min Volatilidad | Volatilidad Media | Máximo Riesgo')
print('  Models: Prophet · ARIMAX · XGBoost · Holt-Winters + Monte Carlo 10k')
print('=' * 80)
print()
print('  ⚠️  DISCLAIMER: Este análisis es exclusivamente educativo.')
print('      No constituye asesoramiento financiero ni recomendación de inversión.')
print('═' * 80)